# Robustness - part 2: can you trust 70 %?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tomasvicar/AUI-public/blob/master/labs/robustness/notebooks/ex5_calibration.ipynb)

**Computer-Aided Medical Diagnostics (MPA-PRM)** · lab for the lecture
*Robustness, generalization and reliability*

The Heart Disease XGBoost of the explainability and fairness labs, the same
recipe. When it says 70 %, do about 70 of 100 such patients have the disease?
Everything is written by hand - the formulas are the lecture's, the code is
yours.

**Three splits, never mixed:** the model is trained on the training part, the
temperature is fitted on the calibration part, and both are judged on the
test part.

**Six holes.** ECE (the expected calibration error) and the Brier score, the
temperature, the comparison, the whole procedure over twenty splits - and
then the same model in three other hospitals: the one number that repairs its probabilities there, and an
honest way to validate a model meant for other hospitals.

# Data, split, model

UCI Heart Disease, Cleveland, 303 patients (CC BY 4.0), stored in the course
repository.

In [ ]:
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xgboost as xgb
from scipy.optimize import minimize_scalar
from scipy.stats import ks_2samp
from sklearn.model_selection import train_test_split

URL = "https://raw.githubusercontent.com/tomasvicar/AUI-public/master/labs/robustness/data/heart_disease_cleveland.csv"
CSV = Path("heart_disease_cleveland.csv")
if not CSV.is_file():
    urllib.request.urlretrieve(URL, CSV)

SEED = 42
N_BINS = 5


def load_heart(path: Path = CSV) -> tuple[pd.DataFrame, pd.Series]:
    """13 features (missing values filled with the median) and disease = num > 0."""
    table = pd.read_csv(path)
    y = (table.pop("num") > 0).astype(int)
    return table.fillna(table.median(numeric_only=True)), y


def split(X: pd.DataFrame, y: pd.Series, seed: int = SEED):
    """Train / calibration / test = 50 / 25 / 25 %, stratified."""
    X_train, X_rest, y_train, y_rest = train_test_split(
        X, y, test_size=0.5, random_state=seed, stratify=y)
    X_calib, X_test, y_calib, y_test = train_test_split(
        X_rest, y_rest, test_size=0.5, random_state=seed, stratify=y_rest)
    return X_train, X_calib, X_test, y_train, y_calib, y_test


def train_xgboost(X_train: pd.DataFrame, y_train: pd.Series, seed: int = SEED):
    """The shared recipe of the tabular running example."""
    model = xgb.XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.1,
                              eval_metric="logloss", random_state=seed)
    return model.fit(X_train, y_train)


def sigmoid(z: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-z))

In [ ]:
X, y = load_heart()
X_train, X_calib, X_test, y_train, y_calib, y_test = split(X, y)
model = train_xgboost(X_train, y_train)
print(f"train {len(X_train)}, calibration {len(X_calib)}, test {len(X_test)} patients")

p_test = model.predict_proba(X_test)[:, 1]
print(f"test accuracy {((p_test > 0.5) == y_test).mean():.3f}")

# The reliability diagram

`reliability_bins` sorts the predictions into five equal-width bins of the
predicted probability and reports, per bin, how many patients fell in, their
mean predicted probability (`confidence`) and the share of them who are sick
(`frequency`). Five bins, not ten: the test part has 76 patients.

### Hole 1 - ECE and the Brier score

In [ ]:
def reliability_bins(p: np.ndarray, y: np.ndarray, n_bins: int = N_BINS) -> list[dict]:
    """Equal-width bins of the predicted probability: count, mean confidence,
    observed frequency of the disease. The first bin includes 0."""
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    index = np.clip(np.digitize(p, edges[1:-1], right=True), 0, n_bins - 1)
    bins = []
    for b in range(n_bins):
        mask = index == b
        bins.append({"low": edges[b], "high": edges[b + 1], "count": int(mask.sum()),
                     "confidence": float(p[mask].mean()) if mask.any() else None,
                     "frequency": float(y[mask].mean()) if mask.any() else None})
    return bins


def plot_reliability(ax, p, y, title):
    """Bars: observed frequency per bin; red: mean predicted probability."""
    ax.plot([0, 1], [0, 1], "k--", lw=1)
    for b in reliability_bins(p, y):
        if b["count"]:
            center = (b["low"] + b["high"]) / 2
            ax.bar(center, b["frequency"], width=0.17, color="teal")
            ax.plot([center - 0.08, center + 0.08], [b["confidence"]] * 2, color="red", lw=2.5)
            ax.text(center, 1.02, f"n={b['count']}", ha="center", fontsize=8)
    ax.set(xlim=(0, 1), ylim=(0, 1.1), xlabel="predicted probability", title=title)


for b in reliability_bins(p_test, y_test.to_numpy()):
    print(b)

In [ ]:
# TODO 1: the two numbers of the lecture, from their formulas.
#
#   ECE   = sum over bins of  count / n * |frequency - confidence|
#           (use reliability_bins; skip the empty bins)
#   Brier = mean of (p - y)^2

def ece(p, y, n_bins=N_BINS):
    ...


def brier(p, y):
    ...


y_test_arr = y_test.to_numpy()
print(f"ECE {ece(p_test, y_test_arr):.3f}, Brier {brier(p_test, y_test_arr):.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 4.2))
plot_reliability(ax, p_test, y_test_arr, "before calibration")
ax.set_ylabel("observed frequency of disease")
plt.show()

# Temperature scaling

One parameter: divide the logit by $T$ before the sigmoid. $T > 1$ softens
the probabilities, $T < 1$ sharpens them.

### Hole 2 - fit T on the calibration part

In [ ]:
# TODO 2: fit the temperature on the CALIBRATION part.
#
# The logit of XGBoost is its raw margin: model.predict(X, output_margin=True).
# Write the negative log-likelihood of p = sigmoid(z / T) - clip p to
# [1e-7, 1 - 1e-7] before the logarithm - and minimize it over T in
# (0.05, 10) with scipy.optimize.minimize_scalar(..., method="bounded").

def fit_temperature(logits, y):
    ...


T = fit_temperature(model.predict(X_calib, output_margin=True), y_calib.to_numpy())
print(f"T = {T:.3f}")

### Hole 3 - judge it on the test part

In [ ]:
# TODO 3: apply T to the TEST logits and compare - ECE, Brier and accuracy
#         before and after. Then draw both reliability diagrams side by side
#         with plot_reliability.

p_after = ...

# One split is not enough

76 test patients in five bins: some bins hold five or six people, and one
patient more or less moves a bar by 0.2. Before you believe a change of ECE,
repeat the whole procedure.

### Hole 4 - twenty splits

In [ ]:
# TODO 4: the whole procedure twenty times, with seeds 0 ... 19:
#         split, train, fit T on the calibration part, ECE and Brier before and
#         after on the test part. How often does ECE improve? And Brier?

rows = []
for seed in range(20):
    ...

# A different hospital

The Heart Disease data were collected in four hospitals in 1988: Cleveland
Clinic, the Hungarian Institute of Cardiology in Budapest, the University
Hospitals of Zürich and Basel, and the VA Medical Center in Long Beach (VA -
US Department of Veterans Affairs). Your model saw only Cleveland. What does
it do elsewhere?

Look at the empty fields as well: the other hospitals did not measure
everything, and Switzerland recorded every cholesterol as 0 - "not measured"
(`load_site_raw` turns it into a missing value).

In [ ]:
# The same data set was collected in four hospitals. The other three files are
# in the course repository as well - downloaded once.
for url in ["https://raw.githubusercontent.com/tomasvicar/AUI-public/master/labs/robustness/data/heart_disease_hungarian.csv",
            "https://raw.githubusercontent.com/tomasvicar/AUI-public/master/labs/robustness/data/heart_disease_switzerland.csv",
            "https://raw.githubusercontent.com/tomasvicar/AUI-public/master/labs/robustness/data/heart_disease_va_long_beach.csv"]:
    name = url.rsplit("/", 1)[1]
    if not Path(name).is_file():
        urllib.request.urlretrieve(url, name)
DATA = Path(".")

SITES = {"Cleveland": "heart_disease_cleveland.csv",
         "Hungary": "heart_disease_hungarian.csv",
         "Switzerland": "heart_disease_switzerland.csv",
         "VA Long Beach": "heart_disease_va_long_beach.csv"}
OTHER_SITES = ["Hungary", "Switzerland", "VA Long Beach"]


def load_site_raw(site: str) -> tuple[pd.DataFrame, pd.Series]:
    """One hospital as recorded: missing values left as NaN. A cholesterol or a
    resting blood pressure of 0 means "not measured" (all of Switzerland, 49
    VA patients) and becomes NaN as well."""
    table = pd.read_csv(DATA / SITES[site])
    y = (table.pop("num") > 0).astype(int)
    for column in ("chol", "trestbps"):
        table.loc[table[column] == 0, column] = np.nan
    return table, y


def cleveland_medians() -> pd.Series:
    return load_site_raw("Cleveland")[0].median()


def auc(p: np.ndarray, y: np.ndarray) -> float:
    """Area under the ROC curve: the probability that a random sick patient
    scores above a random healthy one (ties count one half)."""
    sick, healthy = p[y == 1], p[y == 0]
    greater = (sick[:, None] > healthy[None, :]).mean()
    ties = (sick[:, None] == healthy[None, :]).mean()
    return float(greater + 0.5 * ties)

In [ ]:
# The Cleveland model, with the T of hole 2, in all four hospitals. The gaps of
# the other hospitals are filled with the CLEVELAND medians - the deployed
# pipeline knows only the data it was built on.
medians = cleveland_medians()
logits_sites, y_sites = {}, {}
for site in SITES:
    if site == "Cleveland":
        Xs, ys = X_test, y_test
    else:
        raw, ys = load_site_raw(site)
        Xs = raw.fillna(medians)
    logits_sites[site] = model.predict(Xs, output_margin=True) / T
    y_sites[site] = ys.to_numpy()
    p = sigmoid(logits_sites[site])
    yy = y_sites[site]
    print(f"{site:14s} n {len(yy):3d}  sick {yy.mean():.0%}  AUC {auc(p, yy):.3f}  "
          f"accuracy {((p > 0.5) == yy).mean():.3f}  mean p {p.mean():.3f}  ECE {ece(p, yy):.3f}")

### Hole 5 - one number for a new hospital

The ranking may survive the move while the probabilities do not. Re-fit one
number, the intercept $b$ in $\hat p = \sigma(z/T + b)$, on half of the new
hospital's patients, and judge it on the other half.

In [ ]:
# TODO 5: calibration in the large. For each of the three other hospitals:
#   - split its patients in two stratified halves:
#       fit, judge = train_test_split(np.arange(len(yy)), test_size=0.5,
#                                     random_state=SEED, stratify=yy)
#   - on the `fit` half find the shift b of the logit for which the mean of
#     sigmoid(z + b) equals the share of sick patients there
#     (scipy.optimize.brentq on the interval -20 ... 20 finds the root);
#   - on the `judge` half print the accuracy and ECE before and after.
# Store b of each site in `intercepts[site]`.

def recalibrate_intercept(logits, y):
    ...


intercepts = {}
for site in OTHER_SITES:
    z, yy = logits_sites[site], y_sites[site]
    ...

# Random, or a hospital left out?

Pool all 920 patients and split at random, as most papers do - or leave each
hospital out in turn, train on the other three and test on it. Only the
second answers the question "will it work in another hospital?".

In [ ]:
VALIDATION_REPEATS = 5


def pooled_sites() -> tuple[pd.DataFrame, np.ndarray, np.ndarray]:
    """All four hospitals in one table, as recorded (gaps as NaN), with the site of each patient."""
    tables, labels, sites = [], [], []
    for site in SITES:
        X, y = load_site_raw(site)
        tables.append(X)
        labels.append(y.to_numpy())
        sites += [site] * len(y)
    return pd.concat(tables, ignore_index=True), np.concatenate(labels), np.array(sites)


# All 920 patients in one table. Gaps are filled with the medians of the
# TRAINING part of each split.
X_all, y_all, site_of = pooled_sites()


def fit_and_predict(train, test):
    medians = X_all.iloc[train].median()
    m = train_xgboost(X_all.iloc[train].fillna(medians), y_all[train])
    return m.predict_proba(X_all.iloc[test].fillna(medians))[:, 1]


# The usual way: a random stratified 5-fold split of the pooled data, five times.
from sklearn.model_selection import StratifiedKFold
random_split = {site: {"auc": [], "accuracy": []} for site in ["all"] + list(SITES)}
for repeat in range(VALIDATION_REPEATS):
    p = np.zeros(len(y_all))
    for train, test in StratifiedKFold(5, shuffle=True, random_state=repeat).split(X_all, y_all):
        p[test] = fit_and_predict(train, test)
    for site in random_split:
        mask = np.ones(len(y_all), bool) if site == "all" else site_of == site
        random_split[site]["auc"].append(auc(p[mask], y_all[mask]))
        random_split[site]["accuracy"].append(((p[mask] > 0.5) == y_all[mask]).mean())
for site, v in random_split.items():
    print(f"random split, {site:14s} AUC {np.mean(v['auc']):.3f}  accuracy {np.mean(v['accuracy']):.3f}")

### Hole 6 - leave one hospital out

In [ ]:
# TODO 6: leave one hospital out. For each of the four hospitals: train on the
#         other three (`fit_and_predict(train_indices, test_indices)`), predict
#         the left-out one, and store its AUC and accuracy in
#         held_out[site] = {"auc": ..., "accuracy": ...}; print a row
#         per hospital and the mean AUC and accuracy of the four.
#         `site_of` holds the hospital of every patient.

held_out = {}
for site in SITES:
    ...

# Questions

1. Was the model over- or underconfident? What in the value of $T$ tells you?
2. Did the accuracy change after scaling? Could it have?
3. On your one split, did ECE and the Brier score tell the same story? After
   twenty splits, which of the two would you trust on 76 patients - and why?
4. Why would fitting $T$ on the test part make the "after" numbers
   meaningless?
5. In Switzerland the AUC stays above 0.5, but the accuracy falls far below
   it. How can both be true? What did the intercept repair there - and what not?
6. The random split says AUC 0.87, the hospitals left out say 0.76 on
   average. Which number would you give a hospital that wants to buy the
   model - and why is the random split so optimistic for Switzerland?

In [ ]:
# Space for your answers - e.g. the share of sick patients and the mean prediction per site.

---

*The numbers are computed by [`heart.py`](https://github.com/tomasvicar/AUI-public/blob/master/lectures/robustness/code/heart.py) of
the lecture. The notebook is built by
[`code/build_notebooks.py`](https://github.com/tomasvicar/AUI/blob/master/labs/robustness/code/build_notebooks.py) - editing the
`.ipynb` by hand gets overwritten.*